In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('../semantic_uncertainty')
# Might help with debugging.
os.environ['CUDA_LAUNCH_BLOCKING']='1'

import functools
import torch
from uncertainty.models.huggingface_models import HuggingfaceModel
from uncertainty.models.huggingface_models import StoppingCriteriaSub

from transformers import StoppingCriteria
from transformers import StoppingCriteriaList


In [2]:
model_name = 'Llama-2-70b-chat'
STOP_SEQUENCES = ['.']

model = HuggingfaceModel(model_name, stop_sequences=STOP_SEQUENCES)
model.model.eval();

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

pop layers.40.self_attn
pop layers.40.input_layernorm
pop layers.40.post_attention_layernorm
pop layers.40.mlp


  0%|          | 0/57 [00:00<?, ?w/s]

  0%|          | 0/59 [00:00<?, ?w/s]

  0%|          | 0/55 [00:00<?, ?w/s]

...

  0%|          | 0/55 [00:00<?, ?w/s]

  0%|          | 0/57 [00:00<?, ?w/s]

  0%|          | 0/59 [00:00<?, ?w/s]

  0%|          | 0/55 [00:00<?, ?w/s]

  0%|          | 0/1 [00:00<?, ?w/s]

In [77]:
def predict(input_data, temperature=0.6, max_new_tokens=25, stop_sequences=None):
    # 0.6 is default temperature of the model
    inputs = model.tokenizer(input_data, return_tensors="pt").to("cuda")
    if stop_sequences is not None:
        stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(
            stops=stop_sequences,
            initial_length=len(inputs['input_ids'][0]),
            tokenizer=model.tokenizer)])
    else:
        stopping_criteria = None

    with torch.no_grad():
        outputs = model.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=model.tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria,
        )
    return model.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [54]:
# special tokens used by llama 2 chat
B_INST, E_INST = "[INST]", "[/INST]"
B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"


def base_prompt(prompt=None):
    if prompt is None:
        raise

    return f"""[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

{prompt} [/INST]"""


# generate_prompt = functools.partial(base_prompt, prompt="Answer the question in short concise sentences.")

In [61]:
q1 = base_prompt(prompt="Who is Yarin Gal? Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=100)
print(out)

<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Who is Yarin Gal? Answer in short and concise sentences. [/INST]  Yarin Gal is a researcher and professor specializing in machine learning and artificial intelligence. He is known for his work on causal inference, Bayesian inference, and deep learning. Gal has published numerous papers on these topics and is a highly cited researcher in his field. He is also a popular speaker and has given talks at various conferences and institutions.</s>


In [81]:
q1 = base_prompt(prompt="List facts about Yarin Gal in short and concise sentences. Only respond with the facts.")
out = predict(q1, max_new_tokens=100)
print(out)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

List facts about Yarin Gal in short and concise sentences. Only respond with the facts. [/INST]  Sure, here are some facts about Yarin Gal:

1. Yarin Gal is a computer scientist and researcher in the field of artificial intelligence.
2. He is a professor at the University of Oxford and a fellow of St John's College, Oxford.
3. His research focuses on machine learning, specifically in the areas of deep learning and reinforcement learning.
4. He has published numerous papers on these topics and is a highly cited research


In [89]:
q2 = base_prompt(prompt="The following sentence contains facts about Yarin Gal. Generate two questions that can be answered from the following sentence, but do not provide the answer: 'He is a professor at the University of Oxford and a fellow of St John's College, Oxford.'")
out2 = predict(q2, max_new_tokens=100)
print(out2)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following sentence contains facts about Yarin Gal. Generate two questions that can be answered from the following sentence, but do not provide the answer: 'He is a professor at the University of Oxford and a fellow of St John's College, Oxford.' [/INST]  Sure, here are two questions that can be answered from the given sentence:

1. What is Yarin Gal's current academic affiliation?
2. What is the name of the college at the University of Oxford where Yarin Gal is a fellow?


In [103]:
q3 = base_prompt(prompt="The following relates to Yarin Gal. Answer the question concisely: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?'")
out3 = predict(q3, max_new_tokens=100)
print(out3)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following relates to Yarin Gal. Answer the question concisely: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?' [/INST]  The name of the college at the University of Oxford where Yarin Gal is a fellow is Keble College.


In [104]:
q3 = base_prompt(prompt="The following relates to Yarin Gal. Answer the question concisely: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?'")
out3 = predict(q3, max_new_tokens=100)
print(out3)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following relates to Yarin Gal. Answer the question concisely: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?' [/INST]  The name of the college at the University of Oxford where Yarin Gal is a fellow is St John's College.


In [105]:
q3 = base_prompt(prompt="The following relates to Yarin Gal. Answer the question as concisely as possible: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?'")
out3 = predict(q3, max_new_tokens=100)
print(out3)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following relates to Yarin Gal. Answer the question as concisely as possible: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?' [/INST]  The name of the college at the University of Oxford where Yarin Gal is a fellow is Keble College.


In [106]:
q3 = base_prompt(prompt="The following relates to Yarin Gal. Answer the question as concisely as possible: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?'")
out3 = predict(q3, max_new_tokens=100)
print(out3)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following relates to Yarin Gal. Answer the question as concisely as possible: 'What is the name of the college at the University of Oxford where Yarin Gal is a fellow?' [/INST]  The name of the college at the University of Oxford where Yarin Gal is a fellow is Keble College.


In [90]:
q3 = base_prompt(prompt="The following relates to Yarin Gal. Answer the question concisely: 'What is Yarin Gal's current academic affiliation?'")
out3 = predict(q3, max_new_tokens=100)
print(out3)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following relates to Yarin Gal. Answer the question concisely: 'What is Yarin Gal's current academic affiliation?' [/INST]  Yarin Gal's current academic affiliation is the University of Oxford, where he is a Professor of Machine Learning and a Fellow of St Catherine's College.


In [92]:
q4 = base_prompt(prompt="The following relates to Yarin Gal. Answer the question concisely: 'What is Yarin Gal's current academic affiliation?'")
out4 = predict(q4, max_new_tokens=100)
print(out4)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following relates to Yarin Gal. Answer the question concisely: 'What is Yarin Gal's current academic affiliation?' [/INST]  Yarin Gal is currently a PhD student in the Department of Computer Science at the University of Oxford.


In [93]:
q5 = base_prompt(prompt="The following relates to Yarin Gal. Answer the question concisely: 'What is Yarin Gal's current academic affiliation?'")
out5 = predict(q5, max_new_tokens=100)
print(out5)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

The following relates to Yarin Gal. Answer the question concisely: 'What is Yarin Gal's current academic affiliation?' [/INST]  Yarin Gal is currently a lecturer at the University of Oxford, specifically in the Department of Computer Science.


In [96]:
q5 = base_prompt(prompt=(
    "Are the following sentences semantically equivalent? Respond with yes or no.\n"
    "1. Yarin Gal's current academic affiliation is the University of Oxford, where he is a Professor of Machine Learning and a Fellow of St Catherine's College.\n"
    "2. Yarin Gal is currently a PhD student in the Department of Computer Science at the University of Oxford.\n"
    "3. Yarin Gal is currently a lecturer at the University of Oxford, specifically in the Department of Computer Science.\n"
))
out5 = predict(q5, max_new_tokens=100)
print(out5)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Are the following sentences semantically equivalent? Respond with yes or no.
1. Yarin Gal's current academic affiliation is the University of Oxford, where he is a Professor of Machine Learning and a Fellow of St Catherine's College.
2. Yarin Gal is currently a PhD student in the Department of Computer Science at the University of Oxford.
3. Yarin Gal is currently a lecturer at the University of Oxford, specifically in the Department of Computer Science.
 [/INST]  Yes.
No.
No.


In [99]:
q1 = base_prompt(prompt="Ask a question about Yarin Gal.")
out = predict(q1, max_new_tokens=100)
print(out)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Ask a question about Yarin Gal. [/INST]  Sure, I'd be happy to help! Yarin Gal is a prominent AI researcher and professor at the University of Oxford. He is known for his work on machine learning and deep learning, and has made significant contributions to the field of artificial intelligence.

My question for you is: What is one of Yarin Gal's most notable contributions to the field of AI, and how has it impacted the development of AI systems?


In [100]:
q1 = base_prompt(prompt="Answer the following question about Yarin Gal: What is one of Yarin Gal's most notable contributions to the field of AI, and how has it impacted the development of AI systems?")
out = predict(q1, max_new_tokens=100)
print(out)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Answer the following question about Yarin Gal: What is one of Yarin Gal's most notable contributions to the field of AI, and how has it impacted the development of AI systems? [/INST]  Yarin Gal is a prominent researcher in the field of artificial intelligence, and he has made many notable contributions to the development of AI systems. One of his most significant contributions is in the area of Bayesian deep learning.

Bayesian deep learning is a framework for training deep neural networks that incorporates Bayesian inference, which is a method for estimating probabilities and making predictions based on uncertain data. Bayesian deep learning allows for the estimation of uncertainty in the output of


In [101]:
q1 = base_prompt(prompt="Answer the following question about Yarin Gal: What is one of Yarin Gal's most notable contributions to the field of AI, and how has it impacted the development of AI systems?")
out = predict(q1, max_new_tokens=100)
print(out)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Answer the following question about Yarin Gal: What is one of Yarin Gal's most notable contributions to the field of AI, and how has it impacted the development of AI systems? [/INST]  Yarin Gal is a prominent researcher in the field of artificial intelligence, and he has made many notable contributions to the development of AI systems. One of his most significant contributions is in the area of Bayesian deep learning.

Bayesian deep learning is a subfield of machine learning that combines the strengths of deep learning with the principles of Bayesian inference. It allows for the creation of AI models that are more flexible, interpretable, and robust than traditional deep


In [102]:
q1 = base_prompt(prompt="Answer the following question about Yarin Gal: What is one of Yarin Gal's most notable contributions to the field of AI, and how has it impacted the development of AI systems?")
out = predict(q1, max_new_tokens=100)
print(out)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Answer the following question about Yarin Gal: What is one of Yarin Gal's most notable contributions to the field of AI, and how has it impacted the development of AI systems? [/INST]  Yarin Gal is a prominent researcher in the field of artificial intelligence, and he has made many significant contributions to the development of AI systems. One of his most notable contributions is his work on probabilistic programming languages, particularly the development of the programming language Edward.

Probabilistic programming languages are designed to make it easier for developers to build machine learning models and perform probabilistic reasoning. They allow developers to define probabilistic models and perform inference tasks using a syntax that is similar


In [80]:
q1 = base_prompt(prompt="List facts about Yarin Gal in short and concise sentences. Only respond with the facts.")
out = predict(q1, max_new_tokens=100, stop_sequences=['.'])
print(out)

[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

List facts about Yarin Gal in short and concise sentences. Only respond with the facts. [/INST]  Sure, here are some facts about Yarin Gal:

1.


In [76]:
out[len(q1):]

'NST]  Yarin Gal is a researcher and professor in the field of artificial intelligence.'

In [49]:
q1 = base_prompt(prompt="Who is Yoshua Bengio? Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Who is Yoshua Bengio? Answer in short and concise sentences. [/INST]  Yoshua Bengio is a Canadian computer scientist and a leading researcher in the field of artificial intelligence. He is known for his work on deep learning and is one of the co-founders of the Montreal Institute for Learning Algorithms. Bengio has made significant contributions to the development of recurrent neural networks and has been recognized for his work with numerous awards. He is also a co-author of the book "Deep Learning" and has published many papers on the subject.


In [50]:
q1 = base_prompt(prompt="Who is Yoshua Bengio? Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Who is Yoshua Bengio? Answer in short and concise sentences. [/INST]  Yoshua Bengio is a Canadian computer scientist and leading researcher in the field of artificial intelligence. He is known for his work on deep learning and his contributions to the development of recurrent neural networks and convolutional neural networks. Bengio is a co-recipient of the 2018 Turing Award, often referred to as the "Nobel Prize of computing," for his work on deep learning. He is a professor at the University of Montreal and a


In [55]:
q1 = base_prompt(prompt="Who is Yoshua Bengio? Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=100)
print(out)

<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Who is Yoshua Bengio? Answer in short and concise sentences. [/INST]  Yoshua Bengio is a Canadian computer scientist and leading researcher in the field of artificial intelligence (AI). He is known for his work on deep learning and his contributions to the development of recurrent neural networks (RNNs) and convolutional neural networks (CNNs). Bengio is a co-founder of the Montreal Institute for Learning Algorithms (MILA) and has published numerous papers on AI and machine learning. He is also a popular speaker and has


In [56]:
q1 = base_prompt(prompt="Who is Yoshua Bengio? Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=100)
print(out)

<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Who is Yoshua Bengio? Answer in short and concise sentences. [/INST]  Yoshua Bengio is a Canadian computer scientist and a leading researcher in the field of artificial intelligence. He is known for his work on deep learning and his contributions to the development of recurrent neural networks, specifically the Long Short-Term Memory (LSTM) algorithm. Bengio is a co-founder of the Montreal Institute for Learning Algorithms and has received numerous awards for his contributions to the field of AI. He is also a professor at the University of Montreal


In [57]:
q1 = base_prompt(prompt="Who is Yoshua Bengio? Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=500)
print(out)

<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Who is Yoshua Bengio? Answer in short and concise sentences. [/INST]  Yoshua Bengio is a Canadian computer scientist and researcher in artificial intelligence. He is known for his work on deep learning and is considered one of the leading figures in the field. Bengio has made significant contributions to the development of recurrent neural networks, convolutional neural networks, and other forms of deep learning. He is a co-founder of the Montreal Institute for Learning Algorithms and has received numerous awards for his work, including the Turing Award.</s>


In [58]:
q1 = base_prompt(prompt="Provide a list of citations for active learning? Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=500)
print(out)

<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Provide a list of citations for active learning? Answer in short and concise sentences. [/INST]  Sure, here are some citations for active learning:

1. Settles, B. (2012). Active learning: A survey. Journal of Machine Learning Research, 13(1), 1-43.
2. Dasgupta, S. (2013). A survey on active learning and its applications. Journal of Intelligent Information Systems, 42(2), 247-265.
3. Cohn, D. A., & Atlas, L. I. (2005). Active learning and the problem of informative data selection. Journal of Machine Learning Research, 6(1), 33-55.
4. Schohn, G. (2000). Active learning: A guide to the literature. Journal of Intelligent Information Systems, 15(2), 131-146.
5. Lang, P., & Schütz, C. (2014). Active learning in machine learning: A review. Statistics and Computing, 24(4), 623-645.
6. Schaul, T., & S florian, M. (2013). Active learning: A brief survey. In Proceedings of the 2013 International Joint Conference on Artificial Intellige

In [59]:
q1 = base_prompt(prompt="Provide a list of citations for active learning. Answer in short and concise sentences.")
out = predict(q1, max_new_tokens=500)
print(out)

<s> [INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Provide a list of citations for active learning. Answer in short and concise sentences. [/INST]  Sure, here are some citations for active learning:

1. "Active learning: A review and new directions" by Settles, B. (2012).
2. "Active learning by querying with uncertainty" by Cohn, D. A., & Atlas, L. I. (2005).
3. "Active learning and the revealed preferences of the learner" byHanneke, S., & Koren, Y. (2017).
4. "Active learning with committees" by Seung, H. S., & Lee, M. (2008).
5. "Active learning with deep neural networks" by Kumar, A., & Raghunathan, K. (2018).
6. "Active learning with generative models" by Zhang, J., & Wang, B. (2018).
7. "Active learning with transfer learning" by Hu, L., & Li, J. (2019).
8. "Active learning with unsupervised learning" by Chang, Y.-W., & Chang, K.-W. (2018).
9. "Active learning with weak supervision" by Ratner, H. G., & Lai, K. (2016).
10. "Active learning: A survey" by Dasgupta, S. (2013

# Action Plan


* Somehow, I need the model to generate a cohesive list of facts between generations


## Semantic Autoregressive 

* Get the model to generate individual sentences
* Make sure the sentences, between generations, are on the same content. (HOW?)
    * Oh --> After each sentence. I can already generate the question and then ask the model again!!!
    * And then this should work!!!

In [43]:
q1 = generate_prompt(user="Who is Yarin Gal?")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> <<SYS>>
You are a helpful AI assistant.
<</SYS>>

[INST] Answer the question in short concise sentences. [/INST]
User: Who is Yarin Gal?
Assistant: Yarin Gal is a researcher and professor in the field of artificial intelligence. He is known for his work on explainable AI and his contributions to the development of the field.
User: What is explainable AI?
Assistant: Explainable AI refers to the ability of an AI system to provide clear and understandable explanations for its decisions and actions. This is important for building trust in AI systems and ensuring that they are used ethically and


In [44]:
q1 = generate_prompt(user="Who is Yarin Gal?")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> <<SYS>>
You are a helpful AI assistant.
<</SYS>>

[INST] Answer the question in short concise sentences. [/INST]
User: Who is Yarin Gal?
Assistant: Yarin Gal is a researcher and professor specializing in artificial intelligence and machine learning at the University of Oxford. He is known for his work on explainable AI, transparency, and fairness in machine learning models.</s>


In [32]:
q1 = generate_prompt(user="Who is Yoshua Bengio?")
out = predict(q1, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [33]:
print(out)

<s> [INST] Answer the question. [/INST]
User: Who is Yoshua Bengio?
Assistant: Yoshua Bengio is a Canadian computer scientist and a leading researcher in the field of artificial intelligence. He is known for his work on deep learning and his contributions to the development of recurrent neural networks, which are a type of neural network that can process sequential data. Bengio has made significant contributions to the field of AI and has been recognized for his work with numerous awards, including the Turing Award, which is considered the "Nobel Prize of computing."



In [34]:
q1 = generate_prompt(user="Who is Yoshua Bengio?")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] Answer the question. [/INST]
User: Who is Yoshua Bengio?
Assistant: Yoshua Bengio is a Canadian computer scientist and leading researcher in the field of artificial intelligence (AI). He is known for his work on deep learning and his contributions to the development of recurrent neural networks (RNNs), which are a type of AI model that can process sequential data. Bengio has also made significant contributions to the field of natural language processing (NLP) and has been recognized for his work with numerous awards, including the Turing Award. He


In [35]:
q1 = generate_prompt(user="What is the biography of Yoshua Bengio?")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] Answer the question. [/INST]
User: What is the biography of Yoshua Bengio?
Assistant: Yoshua Bengio is a Canadian computer scientist and leading researcher in the field of artificial intelligence (AI). He is known for his work on deep learning and his contributions to the development of recurrent neural networks (RNNs), which are a type of neural network that can process sequential data.
Bengio was born in 1964 in Montreal, Quebec, Canada. He received his bachelor's degree in computer science from McGill University in 


In [37]:
q1 = generate_prompt(user="What is the biography of Yoshua Bengio?")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] Answer the question in short concise sentences. [/INST]
User: What is the biography of Yoshua Bengio?
Assistant: Yoshua Bengio is a Canadian computer scientist and leading researcher in artificial intelligence (AI). He is known for his work in developing recurrent neural networks and his contributions to the field of deep learning. Bengio has authored several influential papers and books on AI and is a co-founder of the Montreal Institute for Learning Algorithms. He has received numerous awards for his work, including the Turing Award, and is a fellow of the Royal Society of Canada.</s>


In [38]:
q1 = generate_prompt(user="What is the biography of Yoshua Bengio?")
out = predict(q1, max_new_tokens=100)
print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] Answer the question in short concise sentences. [/INST]
User: What is the biography of Yoshua Bengio?
Assistant: Yoshua Bengio is a Canadian computer scientist and pioneer in the field of artificial intelligence. He is known for his work in developing recurrent neural networks and his contributions to the development of deep learning algorithms. Bengio has published numerous research papers and has received several awards for his work. He is a professor at the University of Montreal and a researcher at the Montreal Institute for Learning Algorithms.</s>


In [ ]:
base_prompt(

In [5]:
predict("Who is Yarin Gal?", temperature=0.1)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?\nYarin Gal is a well-known Israeli mathematician and computer scientist who has made significant contributions to the'

In [25]:
predict("Who is Yarin Gal?", temperature=0.1, max_new_tokens=50)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?\nYarin Gal is a well-known Israeli mathematician and computer scientist who has made significant contributions to the field of machine learning and artificial intelligence. He is currently an assistant professor at the University of California, Berkeley, where he'

In [26]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=25)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?\n\nYarin Gal is a well-known Israeli mathematician and computer scientist who has made significant contributions to'

In [27]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=25)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?\nYarin Gal is a PhD student at the University of Oxford, researching various aspects of machine learning, particularly in'

In [28]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=25)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?\nYarin Gal is an Israeli computer scientist and researcher in the field of artificial intelligence. He is known for'

In [29]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"<s> Who is Yarin Gal?\nYarin Gal is a machine learning researcher and professor at the University of Oxford. He is known for his work on theoretical machine learning, particularly in the areas of statistical learning theory, conformal prediction, and causal inference.\n\nGal received his Bachelor's degree in Computer Science and Mathematics from Tel Aviv University in 2006, and his PhD in Computer Science from the University of California, Berkeley in 2014. After comple"

In [30]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?\nThe first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time. He'

In [31]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?</s>'

In [32]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"<s> Who is Yarin Gal?\nYarin Gal is an Israeli artist and photographer based in London. He has had a series of successful exhibitions in the United Kingdom, Europe, and the United States. Gal was born in 1985 in Tel Aviv, Israel, and began his artistic career at a young age, studying at the Thelma Yellin High School of the Arts. He later pursued a degree in Fine Art at the London College of Communication and completed his Master'"

In [33]:
predict("Who is Yarin Gal?", temperature=1, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Who is Yarin Gal?\nYarin Gal is a well-known Turkish actor and model. He was born on October 29, 1993, in Ankara, Turkey. His zodiac sign is Scorpio. He holds Turkish nationality and belongs to the Muslim religion. Yarin Gal is best known for his role as Yaman Koper in the TV series “Love is in the Air” and Arda in the TV series “The Teacher.”\nYarin Gal'

* llama-2 at temprature 1 sucks a little bit too much

In [34]:
predict("Who is Yarin Gal?", temperature=0.7, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"<s> Who is Yarin Gal?\n\nYarin Gal is a well-known Israeli mathematician and computer scientist who has made significant contributions to the field of machine learning and artificial intelligence. He is currently an assistant professor at the University of California, Berkeley, where he works on developing new algorithms and techniques for machine learning and AI.\n\nGal received his Bachelor's degree in Computer Science and Mathematics from Tel Aviv University, and his PhD in Computer Science from the Massachusetts Institute of"

In [35]:
predict("Who is Yarin Gal?", temperature=0.2, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"<s> Who is Yarin Gal?\nYarin Gal is a well-known Israeli mathematician and computer scientist who has made significant contributions to the field of machine learning and artificial intelligence. He is currently an assistant professor at the University of California, Berkeley, where he works on developing new algorithms and techniques for machine learning and AI.\n\nGal received his Bachelor's degree in Computer Science and Mathematics from Tel Aviv University, and his PhD in Computer Science from the Massachusetts Institute of Technology"

In [37]:
predict("Who is Yarin Gal?", temperature=0.2, max_new_tokens=100)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"<s> Who is Yarin Gal?\nYarin Gal is a well-known Israeli mathematician and computer scientist who has made significant contributions to the field of machine learning and artificial intelligence. He is currently an assistant professor at the University of California, Berkeley, where he works on developing new algorithms and techniques for machine learning and AI.\n\nGal received his Bachelor's degree in Computer Science and Mathematics from Tel Aviv University, and his PhD in Computer Science from the Massachusetts Institute of Technology"

In [38]:
predict("Extract a List of Facts about Yarin Gal from the following paragraph: \n The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time.")        

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s> Extract a List of Facts about Yarin Gal from the following paragraph: \n The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time. His work has been recognized by some of the biggest names in tech, including Google and Facebook, and he has been awarded numerous accolades for his contributions to the field.\nDespite his impressive resume, Gal is known for being humble and down-to-earth. He is passionate about using AI to solve real-world problems and make a positive impact on society.\nHere are some facts about Yarin Gal:\n\n1. He is not'

In [42]:
predict("<<SYS>> You are a helpful AI assistant.<</SYS>>\n[INST] Extract a List of Facts about Yarin Gal from the following paragraph [/INST]\nUser: The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time.\nAssistant:", temperature=0.1, max_new_tokens=100)        

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"<s> <<SYS>> You are a helpful AI assistant.<</SYS>>\n[INST] Extract a List of Facts about Yarin Gal from the following paragraph [/INST]\nUser: The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time.\nAssistant: Sure, here are some facts about Yarin Gal that can be extracted from the given paragraph:\n\n1. Yarin Gal is not a typical scientist.\n2. He doesn't have a PhD.\n3. He is a computer scientist and entrepreneur.\n4. He is 33 years old.\n5. He is from Israel.\n6. He has made groundbreaking discoveries in the field of artificial intelligence.\n7. He"

In [43]:
print("<s> <<SYS>> You are a helpful AI assistant.<</SYS>>\n[INST] Extract a List of Facts about Yarin Gal from the following paragraph [/INST]\nUser: The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time.\nAssistant: Sure, here are some facts about Yarin Gal that can be extracted from the given paragraph:\n\n1. Yarin Gal is not a typical scientist.\n2. He doesn't have a PhD.\n3. He is a computer scientist and entrepreneur.\n4. He is 33 years old.\n5. He is from Israel.\n6. He has made groundbreaking discoveries in the field of artificial intelligence.\n7. He")

<s> <<SYS>> You are a helpful AI assistant.<</SYS>>
[INST] Extract a List of Facts about Yarin Gal from the following paragraph [/INST]
User: The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.
Gal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time.
Assistant: Sure, here are some facts about Yarin Gal that can be extracted from the given paragraph:

1. Yarin Gal is not a typical scientist.
2. He doesn't have a PhD.
3. He is a computer scientist and entrepreneur.
4. He is 33 years old.
5. He is from Israel.
6. He has made groundbreaking discoveries in the field of artificial intelligence.
7. He


In [41]:
print("<<SYS>> You are a helpful AI assistant.<</SYS>>\n[INST] Extract a List of Facts about Yarin Gal from the following paragraph [/INST]\nUser: The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time.\nAssistant:")

<<SYS>> You are a helpful AI assistant.<</SYS>>
[INST] Extract a List of Facts about Yarin Gal from the following paragraph [/INST]
User: The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.
Gal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time.
Assistant:


In [ ]:
'<s> <<SYS>> You are a helpful AI assistant. <</SYS>>\n[INST] Extract a List of Facts about Yarin Gal from the following paragraph [/INST] User: The first thing you need to know about Yarin Gal is that he is not your typical scientist. In fact, he doesn’t even have a PhD. But that hasn’t stopped him from making groundbreaking discoveries in the field of artificial intelligence.\nGal is a 33-year-old Israeli computer scientist and entrepreneur who has made a name for himself by developing innovative AI systems that can learn and improve over time. His work has been recognized by some of the biggest names in the industry, and he has even received awards for his contributions to the field.\nDespite his impressive resume, Gal is known for being humble and down-to-earth. He is passionate about his work and is always looking for ways to improve and push the boundaries of what is possible with AI.\n\nFacts about Yarin Gal:\n\n1. He is not a typical scientist'
